In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [106]:
df = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp']).drop(columns=['timestamp'])  # Load your dataset here
df.head()

,user_id,item_id,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [107]:
df.shape

(100000, 3)

In [108]:
df["user_id"] = df["user_id"]-1  # Adjust user_id to be zero-indexed
df["item_id"] = df["item_id"]-1  # Adjust item_id to be zero-indexed

In [109]:
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

In [110]:
class MovieLen100k(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        user_id = self.data.iloc[idx]['user_id']
        item_id = self.data.iloc[idx]['item_id']
        rating = self.data.iloc[idx]['rating']
        return torch.tensor(user_id, dtype=torch.long), torch.tensor(item_id, dtype=torch.long), torch.tensor(rating, dtype=torch.float)

In [111]:
train_data, test_data = MovieLen100k(train_set), MovieLen100k(test_set)
train_loader, test_loader = DataLoader(train_data, batch_size=64, shuffle=True), DataLoader(test_data, batch_size=64, shuffle=False)

In [112]:
global_mean = train_set.rating.mean()
global_mean

np.float64(3.5312625)

In [113]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_size):
        super(MatrixFactorization, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_size)
        self.item_embedding = nn.Embedding(num_items, embedding_size)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_mean = global_mean
    def forward(self, user_id, item_id):
        user_vector = self.user_embedding(user_id)
        item_vector = self.item_embedding(item_id)
        user_bias = self.user_bias(user_id).squeeze()
        item_bias = self.item_bias(item_id).squeeze()
        return (user_vector * item_vector).sum(1) + user_bias + item_bias + self.global_mean # Dot product

In [114]:
model = MatrixFactorization(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), embedding_size=32)

In [115]:
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss = nn.MSELoss()

In [116]:
Epochs = 20  # Set the number of epochs
for epoch in range(Epochs):
    model.train()
    total_loss = 0
    for user_id, item_id, rating in train_loader:
        optimizer.zero_grad()
        predictions = model(user_id, item_id)
        l = loss(predictions, rating)
        l.backward()
        optimizer.step()
        total_loss += l.item()
    print(f'Epoch {epoch+1}/{Epochs}, Loss: {np.sqrt(total_loss/len(train_loader))}')

    model.eval()
    total_loss = 0
    with torch.no_grad():
        for user_id, item_id, rating in test_loader:
            predictions = model(user_id, item_id)
            l = loss(predictions, rating)
            total_loss += l.item()
    print(f'Test Loss: {np.sqrt(total_loss/len(test_loader))}')

Epoch 1/20, Loss: 4.042550214669081
Test Loss: 2.5488487233947863
Epoch 2/20, Loss: 1.5871912513323556
Test Loss: 1.8438618030590206
Epoch 3/20, Loss: 1.0834270065059117
Test Loss: 1.5980505263289355
Epoch 4/20, Loss: 0.9596511759912825
Test Loss: 1.4823844199793643
Epoch 5/20, Loss: 0.9686438152938865
Test Loss: 1.418267798063833
Epoch 6/20, Loss: 0.9825094123200204
Test Loss: 1.3380050444075233
Epoch 7/20, Loss: 0.9302320030478829
Test Loss: 1.2743085944692967
Epoch 8/20, Loss: 0.8681442611787633
Test Loss: 1.2374986585460281
Epoch 9/20, Loss: 0.8338752568707295
Test Loss: 1.2269609366603647
Epoch 10/20, Loss: 0.8160196136625475
Test Loss: 1.206105494007293
Epoch 11/20, Loss: 0.8003092583200028
Test Loss: 1.2018120460174915
Epoch 12/20, Loss: 0.7850319969587036
Test Loss: 1.1870606943088757
Epoch 13/20, Loss: 0.7719106840310013
Test Loss: 1.1806601815965898
Epoch 14/20, Loss: 0.7627160884748065
Test Loss: 1.177146500331675
Epoch 15/20, Loss: 0.7547277437935997
Test Loss: 1.1790732941

In this notebook, we're doing explicit feedback (rating from 1-5) with MF. Since we're predicting 1-5 aka regression, we'll be using MSE